In [ ]:
"""
EC Supplementary Figure 9 — Individual Area Axon Length vs ML Position
=========================================================================
Grid of mini-panels (wraps every 5) showing raw axon length vs distance
to the most medial cell, for all areas, ordered PM -> AT -> Other (order
only — no colour-coded group bands/labels).

Each panel is a mini joint-plot:
  - Light blue dots (individual neurons) + black regression line + 95% CI
  - Top marginal: KDE of ML position (distance to most medial cell)
  - Right marginal: KDE of axon length for that area
  - Title: bold area name, then an italic line with Spearman rho and
    mutual information (MI) between ML position and axon length; stars
    denote the MI permutation-test significance

Groups (order only, as specified):
  PM    : VISpor, RSP, PAR, PRE, PTLp, VIS, POST
  AT    : PERI, ORB, ECT, TEa, AI, PFC (IL+PL), OLF
  Other : ACA, MO, GU, VISC, SS, AUD

HOW TO RUN IN GOOGLE COLAB
  1. Upload this script and master_detailed_comment_updated__2_.csv
  2. Uncomment the google.colab block (lines ~39-43)
  3. Comment out the local path line (line ~46)
  4. Run -> saves EC_individual_areas_supp.pdf and .png

REQUIREMENTS
  pip install pandas numpy matplotlib scipy scikit-learn
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.stats import t as t_dist, gaussian_kde
from sklearn.feature_selection import mutual_info_classif
import io

# ─── 1. LOAD DATA ─────────────────────────────────────────────────────────────

# --- GOOGLE COLAB: uncomment these 4 lines and comment out the local line below ---
# from google.colab import files
# print("Please upload master_detailed_comment_updated__2_.csv")
# uploaded = files.upload()
# fname = next(iter(uploaded))
# df_raw = pd.read_csv(io.BytesIO(uploaded[fname]), encoding='latin-1')

# --- LOCAL: set your file path here ---
df_raw = pd.read_csv('master_detailed_comment.csv', encoding='latin-1')

df_raw.columns = [str(c).strip() for c in df_raw.columns]

# ─── 2. FILTER DATA ───────────────────────────────────────────────────────────
df = df_raw[df_raw['neuron_ID'] != '1807141_24'].copy()
df['proj'] = df['proj'].replace('', np.nan).replace(r'^\s*$', np.nan, regex=True)
df = df.dropna(subset=['proj'])
for c in ['x', 'y', 'z']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = df.dropna(subset=['x', 'y', 'z'])
for col in [c for c in df_raw.columns if c.endswith('_length')]:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# ─── 3. REQUIRE NEOCORTICAL PROJECTIONS ───────────────────────────────────────
# Kept broad for the "any neocortical projection" inclusion criterion —
# this list is deliberately independent of the PM/AT/Other display grouping
# below, so adding AUD or moving areas between display groups doesn't
# change which neurons are included.
NEOCORTICAL_LEN = ['PERI_length', 'ORB_length', 'AI_length', 'OLF_length',
                    'GU_length', 'VISC_length', 'TEa_length', 'ECT_length',
                    'SS_length', 'PFC_length', 'VISpor_length', 'RSP_length',
                    'PTLp_length', 'ACA_length', 'PAR_length', 'POST_length',
                    'PRE_length', 'VIS_length', 'MO_length', 'AUD_length']
present_neo = [c for c in NEOCORTICAL_LEN if c in df.columns]
df['Total_neo'] = df[present_neo].sum(axis=1)
plot_df = df[df['Total_neo'] > 0].copy()

print(f"n = {len(plot_df)} neurons")
print(f"RSP: {(plot_df['proj']=='rsp').sum()}, "
      f"RSP+ORB: {(plot_df['proj']=='rsp_orb').sum()}, "
      f"ORB: {(plot_df['proj']=='orb').sum()}")

z = plot_df['z'].values   # mediolateral coordinate

# ─── 4. DEFINE AREAS AND DISPLAY GROUPS ───────────────────────────────────────
# Each entry: (length_column, display_label, group)
# group in {'pm', 'at', 'other'} controls left-to-right ordering and the
# colour band each panel sits under — panels are NOT split into separate
# structural rows by group, they run in one sequence PM -> AT -> Other.
PM_AREAS = [
    ('VISpor_length', 'VISpor'),
    ('RSP_length',    'RSP'),
    ('PAR_length',    'PAR'),
    ('PRE_length',    'PRE'),
    ('PTLp_length',   'PTLp'),
    ('VIS_length',    'VIS'),
    ('POST_length',   'POST'),
]
AT_AREAS = [
    ('PERI_length', 'PERI'),
    ('ORB_length',  'ORB'),
    ('ECT_length',  'ECT'),
    ('TEa_length',  'TEa'),
    ('AI_length',   'AI'),
    ('PFC_length',  'PFC'),   # IL + PL
    ('OLF_length',  'OLF'),
]
OTHER_AREAS = [
    ('ACA_length',  'ACA'),
    ('MO_length',   'MO'),
    ('GU_length',   'GU'),
    ('VISC_length', 'VISC'),
    ('SS_length',   'SS'),
    ('AUD_length',  'AUD'),
]

ALL_AREAS = ([(c, l, 'pm') for c, l in PM_AREAS] +
             [(c, l, 'at') for c, l in AT_AREAS] +
             [(c, l, 'other') for c, l in OTHER_AREAS])

# ─── 5. COMPUTE STATISTICS ─────────────────────────────────────────────────────
# Spearman rho: monotonic association between ML position (continuous) and
# projection STRENGTH (raw axon length, continuous) — captures functional
# form of the gradient.
#
# Mutual information: statistical dependence between ML position
# (continuous) and the binary ROUTING decision (projects / does not
# project to that area, i.e. axon length > 0) — makes no assumption about
# functional form. Computed with sklearn's mutual_info_classif (Kraskov-
# style k-NN estimator, k=3). Significance is assessed by a permutation
# test: ML position is shuffled across neurons (n_perm=500, independently
# per area) and MI recomputed each time; the p-value is the proportion of
# permuted MI values exceeding the observed MI.
N_PERM = 500
MI_K = 3
rng = np.random.default_rng(0)

def mi_with_permutation_test(z_vals, binary_y, n_perm=N_PERM, k=MI_K):
    """Return (observed_mi, perm_p) for MI(z_vals, binary_y)."""
    z_col = z_vals.reshape(-1, 1)
    if len(np.unique(binary_y)) < 2:
        # No variation in the binary label (all/no neurons project here) —
        # MI is undefined/zero and there is nothing to permute against.
        return 0.0, 1.0

    obs_mi = mutual_info_classif(
        z_col, binary_y, discrete_features=False,
        n_neighbors=k, random_state=0
    )[0]

    perm_mi = np.empty(n_perm)
    for i in range(n_perm):
        z_shuff = rng.permutation(z_vals)
        perm_mi[i] = mutual_info_classif(
            z_shuff.reshape(-1, 1), binary_y, discrete_features=False,
            n_neighbors=k, random_state=0
        )[0]

    perm_p = np.mean(perm_mi > obs_mi)
    return obs_mi, perm_p

area_stats = []
print(f"\n{'Area':<10} {'Grp':<6} {'rho':>7} {'p':>10} {'sig':>5} "
      f"{'MI':>7} {'MI_p':>8} {'MI_sig':>7}")
print("-" * 65)

for col, label, grp in ALL_AREAS:
    if col not in plot_df.columns:
        print(f"  [skip] {label} — column {col} not found")
        continue
    vals = plot_df[col].values
    rho, p = stats.spearmanr(z, vals)
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))

    binary_proj = (vals > 0).astype(int)
    mi, mi_p = mi_with_permutation_test(z, binary_proj)
    mi_sig = '***' if mi_p < 0.001 else ('**' if mi_p < 0.01 else ('*' if mi_p < 0.05 else 'ns'))

    print(f"  {label:<10} {grp:<6} {rho:>7.3f} {p:>10.4f} {sig:>5} "
          f"{mi:>7.3f} {mi_p:>8.4f} {mi_sig:>7}")
    area_stats.append({'col': col, 'label': label, 'grp': grp,
                        'rho': rho, 'p': p, 'sig': sig,
                        'mi': mi, 'mi_p': mi_p, 'mi_sig': mi_sig})

# Order within each group: PM by ascending rho (most negative gradient
# first), AT by descending rho (most positive first), Other by |rho|
# descending. Groups themselves stay in fixed PM -> AT -> Other order.
pm_sorted    = sorted([a for a in area_stats if a['grp'] == 'pm'],    key=lambda x: x['rho'])
at_sorted    = sorted([a for a in area_stats if a['grp'] == 'at'],    key=lambda x: -x['rho'])
other_sorted = sorted([a for a in area_stats if a['grp'] == 'other'], key=lambda x: -abs(x['rho']))

ordered_areas = pm_sorted + at_sorted + other_sorted
n_panels = len(ordered_areas)

# ─── 6. COLOUR SCHEME ─────────────────────────────────────────────────────────
DOT_COLOR    = '#a8c8dc'   # light blue — dots (matches reference figure)
LINE_COLOR   = '#666666'   # grey — regression line (kept subtle vs. dots)
CI_COLOR     = '#888888'   # grey — CI shading
MARGIN_COLOR = '#a8c8dc'   # marginal density fill (matches dots)
TITLE_COLOR  = '#000000'   # plain black titles — groupings are kept as an
                            # ordering only (PM -> AT -> Other), not called
                            # out with colour bands/labels

N_COLS = 4   # panels per row in the grid — 4x5 grid sized for A4 printing

# ─── 7. PLOT STYLE ────────────────────────────────────────────────────────────
# Arial isn't bundled with this environment's font cache — Liberation Sans
# is metrically identical and matplotlib will fall back to it automatically
# when 'Arial' is requested but unavailable.
BG = 'white'
plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Arial', 'Liberation Sans', 'DejaVu Sans'],
    'font.size':          12,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.linewidth':     0.8,
    'axes.edgecolor':     '#333333',
    'xtick.color':        '#333333',
    'ytick.color':        '#333333',
    'xtick.labelsize':    10.5,
    'ytick.labelsize':    10.5,
    'figure.facecolor':   BG,
    'axes.facecolor':     'white',
})

# ─── 8. DISPLAY-COORDINATE PREP ───────────────────────────────────────────────
# Shift ML position so the most medial cell sits at 0 (matches the
# "distance to most medial cell" axis convention), and convert axon length
# to mm for display (stats above were computed on the raw µm columns).
z_display = z - z.min()
Z_MAX = np.ceil(z_display.max() / 100) * 100   # shared x-limit, nice round number

# ─── 9. PANEL DRAWING FUNCTION (mini joint-plot) ──────────────────────────────
def draw_panel(fig, outer_spec, area, show_ylabel=False):
    """
    Draw one mini joint-plot panel into outer_spec: main scatter +
    regression (bottom-left), top marginal KDE of ML position (top-left),
    right marginal KDE of axon length (bottom-right).
    """
    col, label = area['col'], area['label']
    rho, sig, mi, mi_sig = area['rho'], area['sig'], area['mi'], area['mi_sig']

    vals_mm = plot_df[col].values / 1000.0
    p95 = max(np.percentile(vals_mm, 95), vals_mm.max() * 0.01, 1e-6)
    vals_c = np.minimum(vals_mm, p95)

    inner = gridspec.GridSpecFromSubplotSpec(
        2, 2, subplot_spec=outer_spec,
        width_ratios=[4, 1], height_ratios=[1, 4],
        wspace=0.06, hspace=0.10
    )
    ax_main = fig.add_subplot(inner[1, 0])
    ax_top  = fig.add_subplot(inner[0, 0], sharex=ax_main)
    ax_right = fig.add_subplot(inner[1, 1], sharey=ax_main)

    # ── Main scatter ───────────────────────────────────────────────────────
    ax_main.scatter(z_display, vals_c, color=DOT_COLOR, s=8, alpha=0.7,
                     linewidths=0, zorder=2)

    # ── Linear regression + 95% CI ────────────────────────────────────────
    slope, intercept, r, pv, se = stats.linregress(z_display, vals_mm)
    xl = np.linspace(0, Z_MAX, 120)
    yl = np.minimum(slope * xl + intercept, p95)

    n = len(z_display)
    xm = z_display.mean()
    tc = t_dist.ppf(0.975, df=n - 2)
    yp = slope * z_display + intercept
    rse = np.sqrt(np.sum((vals_mm - yp) ** 2) / (n - 2))
    sb = rse * np.sqrt(1 / n + (xl - xm) ** 2 / np.sum((z_display - xm) ** 2))

    ax_main.fill_between(xl,
                          np.minimum(yl - tc * sb, p95),
                          np.minimum(yl + tc * sb, p95),
                          color=CI_COLOR, alpha=0.25, zorder=3)
    ax_main.plot(xl, np.maximum(yl, 0), color=LINE_COLOR, lw=0.8, zorder=4)

    ax_main.set_ylim(0, p95 * 1.08)
    ax_main.set_xlim(0, Z_MAX)
    ax_main.set_xticks([0, Z_MAX])
    ax_main.set_xticklabels(['0', f'{int(Z_MAX)}'], fontsize=10.5)
    y_top = p95 if p95 >= 1 else round(p95, 2)
    ax_main.set_yticks([0, y_top])
    y_lbl = f'{y_top:.0f}' if y_top >= 1 else f'{y_top:.2f}'
    ax_main.set_yticklabels(['0', y_lbl], fontsize=10.5)
    ax_main.tick_params(length=3)

    if show_ylabel:
        ax_main.set_ylabel('Axon length (mm)', fontsize=12.5)

    # ── Top marginal: KDE of ML position ───────────────────────────────────
    ax_top.set_facecolor('none')
    try:
        kde_x = gaussian_kde(z_display)
        xs = np.linspace(0, Z_MAX, 200)
        dens_x = kde_x(xs)
        ax_top.fill_between(xs, dens_x, color=MARGIN_COLOR, alpha=0.4, lw=0)
        ax_top.plot(xs, dens_x, color=MARGIN_COLOR, lw=0.8)
        # Extra headroom above the density curve so the title text (drawn
        # above this axes in figure space) never touches it.
        ax_top.set_ylim(0, dens_x.max() * 2.3)
    except np.linalg.LinAlgError:
        pass
    ax_top.axis('off')

    # ── Right marginal: KDE of axon length (capped at p95 like main) ──────
    ax_right.set_facecolor('none')
    try:
        kde_y = gaussian_kde(vals_c)
        ys = np.linspace(0, p95 * 1.08, 200)
        ax_right.fill_betweenx(ys, kde_y(ys), color=MARGIN_COLOR, alpha=0.4, lw=0)
        ax_right.plot(kde_y(ys), ys, color=MARGIN_COLOR, lw=0.8)
    except np.linalg.LinAlgError:
        pass
    ax_right.axis('off')

    # ── Panel title: bold area name + italic stats line (matches reference) ─
    # Significance stars follow the MI permutation test (the primary novel
    # measure); rho is reported alongside without its own stars, as in the
    # reference figure.
    rho_str = f'{rho:.2f}'
    if rho_str == '-0.00':
        rho_str = '0.00'
    ax_top.text(0.5, 2.05, label, transform=ax_top.transAxes,
                fontsize=13, fontweight='bold', color=TITLE_COLOR,
                ha='center', va='bottom')
    ax_top.text(0.5, 1.35,
                f'ρ = {rho_str}   MI: {mi:.2f} {mi_sig}',
                transform=ax_top.transAxes,
                fontsize=11, style='italic', color=TITLE_COLOR,
                ha='center', va='bottom')

    return ax_main

# ─── 10. BUILD FIGURE (grid, wraps every N_COLS panels; sized for A4 print) ───
n_rows = int(np.ceil(n_panels / N_COLS))

# A4 portrait, inches — one full page, panels sized/spaced to fill it so
# fonts stay legible when printed.
A4_WIDTH, A4_HEIGHT = 8.27, 11.69
fig = plt.figure(figsize=(A4_WIDTH, A4_HEIGHT))
fig.patch.set_facecolor(BG)

outer = gridspec.GridSpec(
    n_rows, N_COLS, figure=fig,
    wspace=0.32, hspace=1.25,
    left=0.09, right=0.97, top=0.93, bottom=0.055
)

for j, area in enumerate(ordered_areas):
    row, col_idx = divmod(j, N_COLS)
    ax_main = draw_panel(fig, outer[row, col_idx], area,
                          show_ylabel=(col_idx == 0))

# ─── 11. SHARED X-AXIS LABEL (once, below the last row) ──────────────────────
last_row = n_rows - 1
last_col_used = min(N_COLS, n_panels - last_row * N_COLS) - 1
pos0 = outer[last_row, 0].get_position(fig)
pos1 = outer[last_row, last_col_used].get_position(fig)
fig.text((pos0.x0 + pos1.x1) / 2, pos0.y0 - 0.028,
          'Distance to most medial cell  (µm)',
          fontsize=12, ha='center', va='top', color='#333333')

# ─── 12. TITLE ─────────────────────────────────────────────────────────────────
fig.suptitle(
    'Axon length vs. distance to most medial cell, by target area',
    fontsize=13, y=0.99, color='#000000')

# ─── 13. SAVE ──────────────────────────────────────────────────────────────────
plt.savefig('EC_individual_areas_supp.pdf', facecolor=BG)
plt.savefig('EC_individual_areas_supp.png', dpi=300, facecolor=BG)
print("\nSaved EC_individual_areas_supp.pdf and .png (A4 portrait)")
plt.show()